# Generating Egyptian Hieroglyphs with Diffusion Models: Normal vs. Bold

This notebook runs the full research pipeline for the assignment and saves every number, table and figure the report needs.

| Report question | What this notebook produces |
|---|---|
| **Q1** How much data is needed? | The same model trained on nested subsets of the signs for a **fixed number of training steps**. KID, FID and memorization are plotted against dataset size. |
| **Q2** Does it work, and are the outputs meaningful? | Real vs. generated grids, nearest-training-image comparisons (novel sign or copy?), shape statistics, training progress, the denoising trajectory, and a blind human-evaluation pack. |
| **Q3** What changed in the model or noise schedule? | Ablations against the baseline: cosine noise schedule, EMA weights, v-prediction, a smaller UNet, 64 px resolution, and the number of sampling steps. |

**Cases.** *Normal* and *Bold* are made from **the same signs**, sharing the same train/test split. The only difference is stroke thickness, so any gap in quality comes from line weight.

**How to run**
1. In Colab, choose Runtime → Change runtime type → **GPU**.
2. Put the chart PDF(s) listed in the configuration cell in `PROJECT_DIR` on Google Drive, or in the Colab working folder.
3. Run all cells. Everything is saved to Google Drive, and finished runs are skipped. If Colab disconnects, run all cells again: training resumes from the last checkpoint.

**One project, several notebooks.** The Hieroglyph notebook and the Cuneiform notebook use the **same** `PROJECT_DIR`. Each one trains only its own cases. The results, figures and tables in Sections 5–9, however, include **every case that has been run so far**. Whichever notebook you run last therefore produces the combined three-case figures for the report.

**Outputs** (inside `PROJECT_DIR`)
```
data/<case>/*.png, data/meta_<script>.csv, data/splits/
runs/<run_name>/   config.json, loss.csv, progress/step_*.png, samples.npz, grid_final.png, nn_pairs.png, metrics.json
figures/           every figure for the report (PNG, 200 dpi)
report/            results.csv, results_table.md, key_numbers.md, dataset_summary_<script>.json, human_eval/
```

## 0. Setup and configuration

In [ ]:
%pip install -q pymupdf pillow diffusers accelerate "torchmetrics[image]" scipy pandas matplotlib imageio

In [ ]:
import os, io, re, json, time, math, copy, random, shutil
from pathlib import Path
from dataclasses import dataclass, asdict, field

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
import torch
import torch.nn.functional as F

# ---------- where everything is saved ----------
USE_DRIVE = True          # keep results on Google Drive so they survive Colab disconnects
PROJECT_NAME = "glyph_diffusion"
PROJECT_DIR = Path(PROJECT_NAME)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as e:
        print("Google Drive not available, saving locally:", e)

DATA_DIR   = PROJECT_DIR / "data"
SPLIT_DIR  = DATA_DIR / "splits"
RUNS_DIR   = PROJECT_DIR / "runs"
FIG_DIR    = PROJECT_DIR / "figures"
REPORT_DIR = PROJECT_DIR / "report"
for d in (DATA_DIR, SPLIT_DIR, RUNS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- dataset extraction (Unicode chart PDF) ----------
PDF_NAME   = "Egyptian Heiroglyphs.pdf"
FIRST_PAGE = 2            # 1-based, inclusive
LAST_PAGE  = 17           # 1-based, inclusive (pages 2-17 = all 3,995 Extended-A signs)
DPI  = 300                # render resolution before resizing
PAD  = 0.8                # inset from grid lines (pt) so borders are not included
SIZE = 128                # stored PNG size (square)
BOLD = 3                  # stroke thickening for the bold case (odd: 3 = bold, 5 = heavy)
MIN_INK = 0.001           # cells with less ink than this are treated as empty and skipped
FORCE_EXTRACT = False     # True = re-extract even if the PNGs already exist

SUBSET_SIZES  = [250, 500, 1000, 2000, None]  # Q1 sweep; None = all training signs (about 3,600)

# ---------- experiment ----------
SEED = 0
TEST_FRACTION = 0.10                          # held-out real signs, used only for evaluation
TRAIN_STEPS   = 10_000    # SAME for every run, so small datasets are not under-trained
BATCH_SIZE    = 32
LEARNING_RATE = 1e-4
WARMUP_STEPS  = 500
SNAPSHOT_EVERY = 1000     # fixed-noise sample grid during training
CKPT_EVERY     = 1000     # resumable checkpoint
N_EVAL_SAMPLES = 500      # generated images per run used for the metrics
EVAL_BATCH     = 50
INFERENCE_STEPS = 50      # DPM-Solver++ sampling steps
EVAL_SEED = 2024          # same starting noise for every run, so grids are directly comparable
SAVE_MODEL = True         # ~310 MB per run (77M-param UNet). False = delete after evaluation (trajectory/steps figures then skipped)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu":
    print("WARNING: no GPU found. Training will be extremely slow. Runtime -> Change runtime type -> GPU")

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10})

def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path)
    print("saved", path)
    plt.show()
    plt.close(fig)

## 1. Build the dataset

### 1a. Extract normal and bold signs in one pass
Each chart cell is rendered once. The **same** crop is saved twice: once as it is (`hiero_normal`) and once thickened (`hiero_bold`). Each case gets its own folder, so the cases never overwrite each other. A metadata row is saved for every sign.

In [ ]:
import pymupdf  # PyMuPDF

HEX_RE = re.compile(r"^[0-9A-F]{4,6}$")

def grid_lines(page, tol=1.0):
    # Return sorted x positions of vertical lines and y positions of horizontal lines.
    xs, ys = set(), set()
    for d in page.get_drawings():
        for item in d["items"]:
            if item[0] == "l":
                p1, p2 = item[1], item[2]
                if abs(p1.x - p2.x) < tol and abs(p1.y - p2.y) > 50:
                    xs.add(round(p1.x, 1))
                elif abs(p1.y - p2.y) < tol and abs(p1.x - p2.x) > 50:
                    ys.add(round(p1.y, 1))
            elif item[0] == "re":
                r = item[1]
                if r.width < tol and r.height > 50:
                    xs.add(round(r.x0, 1))
                elif r.height < tol and r.width > 50:
                    ys.add(round(r.y0, 1))
    def dedupe(vals):
        out = []
        for v in sorted(vals):
            if not out or v - out[-1] > 2:
                out.append(v)
        return out
    return dedupe(xs), dedupe(ys)

def code_labels(page, max_height=8.5):
    # Small hex-code labels printed under each glyph (column headers are larger and skipped).
    labels = []
    for x0, y0, x1, y1, text, *_ in page.get_text("words"):
        if HEX_RE.match(text) and (y1 - y0) < max_height:
            labels.append((pymupdf.Rect(x0, y0, x1, y1), text))
    return labels

def find_cell(rect, xs, ys):
    cx, cy = (rect.x0 + rect.x1) / 2, (rect.y0 + rect.y1) / 2
    left = max((x for x in xs if x <= cx), default=None)
    right = min((x for x in xs if x > cx), default=None)
    top = max((y for y in ys if y <= cy), default=None)
    bottom = min((y for y in ys if y > cy), default=None)
    if None in (left, right, top, bottom):
        return None
    return pymupdf.Rect(left, top, right, bottom)

def to_square(img, size=SIZE):
    # Scale the glyph to fit inside size x size (aspect ratio kept) and centre it on white.
    img = img.convert("L")
    img.thumbnail((size, size), Image.LANCZOS)
    canvas = Image.new("L", (size, size), 255)
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas

def make_bold(img, bold=None):
    # Thicken dark strokes with a min-filter, then smooth the edges.
    bold = globals().get("BOLD", 1) if bold is None else bold
    img = img.convert("L")
    if bold and bold > 1:
        img = img.filter(ImageFilter.MinFilter(bold | 1))
        img = img.filter(ImageFilter.GaussianBlur(0.6))
    return img

def ink_ratio(img, thr=128):
    return float((np.asarray(img) < thr).mean())

def glyph_aspect(img):
    a = np.asarray(img) < 200
    if not a.any():
        return float("nan")
    rows, cols = np.where(a.any(1))[0], np.where(a.any(0))[0]
    return float((cols[-1] - cols[0] + 1) / (rows[-1] - rows[0] + 1))

In [ ]:
NORMAL_DIR = DATA_DIR / "hiero_normal"
BOLD_DIR   = DATA_DIR / "hiero_bold"
META_PATH  = DATA_DIR / "meta_hiero.csv"

def find_pdf(name):
    for p in (Path(name), PROJECT_DIR / name, Path("/content") / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found. Upload it to {PROJECT_DIR} or the Colab working folder.")

if META_PATH.exists() and not FORCE_EXTRACT:
    meta = pd.read_csv(META_PATH, dtype={"code": str})
    print(f"Dataset already extracted: {len(meta)} signs (set FORCE_EXTRACT=True to redo)")
else:
    NORMAL_DIR.mkdir(parents=True, exist_ok=True)
    BOLD_DIR.mkdir(parents=True, exist_ok=True)
    doc = pymupdf.open(find_pdf(PDF_NAME))
    rows, skipped = [], []
    for pno in range(FIRST_PAGE - 1, min(LAST_PAGE, len(doc))):
        page = doc[pno]
        xs, ys = grid_lines(page)
        count = 0
        for label_rect, code_ in code_labels(page):
            cell = find_cell(label_rect, xs, ys)
            if cell is None:
                continue
            clip = pymupdf.Rect(cell.x0 + PAD, cell.y0 + PAD, cell.x1 - PAD, label_rect.y0 - 0.3)
            if clip.is_empty or clip.height < 5:
                continue
            pix = page.get_pixmap(clip=clip, dpi=DPI, alpha=False)
            raw = Image.open(io.BytesIO(pix.tobytes("png"))).convert("L")
            normal, bold = to_square(raw), to_square(make_bold(raw))
            if ink_ratio(normal, thr=200) < MIN_INK:   # empty / reserved cell (loose threshold keeps faint signs)
                skipped.append(code_)
                continue
            normal.save(NORMAL_DIR / f"{code_}.png")
            bold.save(BOLD_DIR / f"{code_}.png")
            rows.append(dict(code=code_, page=pno + 1,
                             ink_normal=ink_ratio(normal), ink_bold=ink_ratio(bold),
                             aspect=glyph_aspect(raw)))
            count += 1
        print(f"page {pno + 1}: {count} signs")
    meta = (pd.DataFrame(rows).drop_duplicates("code")
              .assign(cp=lambda d: d.code.apply(lambda c: int(c, 16)))
              .sort_values("cp").drop(columns="cp").reset_index(drop=True))
    meta.to_csv(META_PATH, index=False)
    print(f"done: {len(meta)} paired signs, skipped {len(skipped)} empty cells {skipped[:20]}")

### 1b. Dataset description (for the report's Data section)

In [ ]:
rng = np.random.default_rng(SEED)
show = rng.choice(meta.code.values, size=min(10, len(meta)), replace=False)
fig, axes = plt.subplots(2, len(show), figsize=(1.3 * len(show), 2.9))
axes = np.array(axes).reshape(2, -1)
for j, c in enumerate(show):
    for i, (d, lab) in enumerate([(NORMAL_DIR, "Normal"), (BOLD_DIR, "Bold")]):
        axes[i, j].imshow(Image.open(d / f"{c}.png"), cmap="gray", vmin=0, vmax=255)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if j == 0: axes[i, j].set_ylabel(lab)
        if i == 0: axes[i, j].set_title(f"U+{c}", fontsize=8)
fig.suptitle("Egyptian hieroglyphs: same signs, two stroke weights")
savefig(fig, "fig_dataset_examples_hiero.png")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
bins = np.linspace(0, max(meta.ink_bold.max(), meta.ink_normal.max()), 40)
ax[0].hist(meta.ink_normal, bins=bins, alpha=.7, label="Normal")
ax[0].hist(meta.ink_bold, bins=bins, alpha=.7, label="Bold")
ax[0].set_xlabel("ink ratio (fraction of dark pixels)"); ax[0].set_ylabel("signs"); ax[0].legend()
ax[1].hist(meta.aspect.dropna().clip(0, 4), bins=40, color="gray")
ax[1].set_xlabel("glyph width / height"); ax[1].set_ylabel("signs")
fig.suptitle("Egyptian hieroglyphs: dataset statistics")
savefig(fig, "fig_dataset_stats_hiero.png")

dataset_summary = {
    "source": PDF_NAME, "pages": f"{FIRST_PAGE}-{LAST_PAGE}",
    "code_range": f"U+{meta.code.iloc[0]}..U+{meta.code.iloc[-1]}",
    "n_signs": int(len(meta)), "image_size": SIZE, "bold_filter_px": BOLD,
    "ink_ratio_normal_mean": float(meta.ink_normal.mean()),
    "ink_ratio_bold_mean": float(meta.ink_bold.mean()),
    "bold_to_normal_ink": float(meta.ink_bold.mean() / meta.ink_normal.mean()),
}
json.dump(dataset_summary, open(REPORT_DIR / "dataset_summary_hiero.json", "w"), indent=2)
pd.Series(dataset_summary)

### 1c. Register the cases and make the train/test split
`ALL_CASES` lists every dataset in the project. A case is used only once its data has been extracted (its metadata file exists), so this notebook also picks up cases that the other notebook has prepared. Cases in the same split group (hieroglyphs Normal and Bold) share the same held-out test signs and the same nested training subsets.

`THIS_NOTEBOOK_CASES` are the cases this notebook **trains**.

In [ ]:
ALL_CASES = {
    "hiero_normal": dict(dir=DATA_DIR / "hiero_normal", split="hiero", label="Hieroglyphs (normal)", meta=DATA_DIR / "meta_hiero.csv"),
    "hiero_bold":   dict(dir=DATA_DIR / "hiero_bold",   split="hiero", label="Hieroglyphs (bold)",   meta=DATA_DIR / "meta_hiero.csv"),
    "cuneiform":    dict(dir=DATA_DIR / "cuneiform",    split="cuneiform", label="Cuneiform",        meta=DATA_DIR / "meta_cuneiform.csv"),
}
CASES = {k: v for k, v in ALL_CASES.items() if v["meta"].exists() and v["dir"].exists()}
THIS_NOTEBOOK_CASES = ['hiero_normal', 'hiero_bold']
assert all(c in CASES for c in THIS_NOTEBOOK_CASES), "Run the extraction cell above first."

def make_split(group, codes):
    # Fixed held-out test set + a random order of training codes.
    # Subset N = the first N codes of that order, so subsets are nested (250 is inside 500, and so on).
    test_p, order_p = SPLIT_DIR / f"{group}_test.txt", SPLIT_DIR / f"{group}_train_order.txt"
    if test_p.exists() and order_p.exists():
        return
    codes = sorted(codes)
    perm = list(np.random.default_rng(SEED).permutation(codes))
    n_test = int(round(len(codes) * TEST_FRACTION))
    test_p.write_text("\n".join(sorted(perm[:n_test])))
    order_p.write_text("\n".join(perm[n_test:]))

def codes_in(case):
    listed = set(pd.read_csv(CASES[case]["meta"], dtype={"code": str}).code)
    return sorted(p.stem for p in CASES[case]["dir"].glob("*.png") if p.stem in listed)

for group in sorted({c["split"] for c in CASES.values()}):
    members = [k for k, v in CASES.items() if v["split"] == group]
    common = set.intersection(*(set(codes_in(k)) for k in members))   # paired cases use only shared codes
    make_split(group, common)

def test_codes(case):
    return (SPLIT_DIR / f"{CASES[case]['split']}_test.txt").read_text().split()

def train_codes(case, n=None):
    order = (SPLIT_DIR / f"{CASES[case]['split']}_train_order.txt").read_text().split()
    return order if n is None else order[:n]

for k in CASES:
    mark = "trained here" if k in THIS_NOTEBOOK_CASES else "from the other notebook"
    print(f"{k:14s} train={len(train_codes(k)):5d}  test={len(test_codes(k)):4d}   ({mark})")

## 2. Model, training and sampling

Compared with the original notebook:
* Training runs for a **fixed number of steps** (`TRAIN_STEPS`), not epochs. With 250 signs, 50 epochs is only about 400 steps, so the dataset-size comparison would really be measuring training length.
* The inference scheduler is built **from the training scheduler's config**. The original used a default `DPMSolverMultistepScheduler()`, which silently mismatches if you change the noise schedule for Q3.
* Loss is logged to CSV, sample grids from fixed noise are saved during training, and checkpoints allow resuming after a disconnect.
* Optional **EMA** weights and **v-prediction** are available for Q3.

In [ ]:
from diffusers import UNet2DModel, DDPMScheduler, DPMSolverMultistepScheduler, DDPMPipeline
from diffusers.optimization import get_cosine_schedule_with_warmup

DOWN = ("DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D")
UP   = ("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D")
MODEL_PRESETS = {
    "base":  dict(layers_per_block=1, block_out_channels=(128, 128, 256, 256, 512, 512)),  # original notebook
    "small": dict(layers_per_block=1, block_out_channels=(64, 64, 128, 128, 256, 256)),
    "deep":  dict(layers_per_block=2, block_out_channels=(128, 128, 256, 256, 512, 512)),
}

@dataclass
class RunCfg:
    case: str
    n_train: int = None              # None = all training signs
    group: str = "Q1"                # Q1 / Q3, used for figures
    variant: str = "baseline"        # short label used in Q3 figures
    image_size: int = SIZE
    model: str = "base"
    beta_schedule: str = "linear"    # "linear" | "squaredcos_cap_v2" (cosine) | "scaled_linear"
    num_train_timesteps: int = 1000
    prediction_type: str = "epsilon" # "epsilon" | "v_prediction"
    ema: bool = False
    ema_decay: float = 0.999
    steps: int = TRAIN_STEPS
    batch_size: int = BATCH_SIZE
    lr: float = LEARNING_RATE
    seed: int = SEED

    @property
    def name(self):
        n = "all" if self.n_train is None else str(self.n_train)
        return f"{self.case}__N{n}__{self.variant}__s{self.seed}"

    @property
    def dir(self):
        return RUNS_DIR / self.name

def build_model(cfg):
    p = MODEL_PRESETS[cfg.model]
    return UNet2DModel(sample_size=cfg.image_size, in_channels=1, out_channels=1,
                       down_block_types=DOWN, up_block_types=UP, **p)

def build_train_scheduler(cfg):
    return DDPMScheduler(num_train_timesteps=cfg.num_train_timesteps,
                         beta_schedule=cfg.beta_schedule, prediction_type=cfg.prediction_type)

def load_images(case, codes, size=SIZE):
    # uint8 tensor (N, H, W); white background = 255.
    out = []
    for c in codes:
        im = Image.open(CASES[case]["dir"] / f"{c}.png").convert("L")
        if im.size != (size, size):
            im = im.resize((size, size), Image.LANCZOS)
        out.append(np.asarray(im))
    return torch.from_numpy(np.stack(out))

def to_model(u8):
    return u8.float().div(127.5).sub(1).unsqueeze(1)

def to_u8(x):
    return ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8).squeeze(1).cpu()

@torch.no_grad()
def sample(unet, sched_config, n, steps=INFERENCE_STEPS, seed=EVAL_SEED, batch=EVAL_BATCH, keep_traj=False):
    # DPM-Solver++ sampling. Returns uint8 (n, H, W) and, if keep_traj, a list of x_t for the first image.
    unet.eval()
    sch = DPMSolverMultistepScheduler.from_config(sched_config)
    size = unet.config.sample_size
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    outs, traj = [], []
    for b in range(0, n, batch):
        bs = min(batch, n - b)
        x = torch.randn((bs, 1, size, size), generator=g, device=DEVICE)
        sch.set_timesteps(steps, device=DEVICE)
        if keep_traj and b == 0:
            traj.append((int(sch.timesteps[0]) + 1, to_u8(x[:1])[0].numpy()))
        for t in sch.timesteps:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = unet(x, t, return_dict=False)[0]
            x = sch.step(out.float(), t, x).prev_sample
            if keep_traj and b == 0:
                traj.append((int(t), to_u8(x[:1])[0].numpy()))
        outs.append(to_u8(x))
    imgs = torch.cat(outs).numpy()
    return (imgs, traj) if keep_traj else imgs

def grid_image(u8, rows, cols, pad=2):
    h, w = u8.shape[1:]
    canvas = np.full((rows * (h + pad) - pad, cols * (w + pad) - pad), 128, np.uint8)
    for i in range(min(len(u8), rows * cols)):
        r, c = divmod(i, cols)
        canvas[r * (h + pad): r * (h + pad) + h, c * (w + pad): c * (w + pad) + w] = u8[i]
    return Image.fromarray(canvas)

def load_unet(cfg):
    unet = UNet2DModel.from_pretrained(cfg.dir / "model", subfolder="unet").to(DEVICE)
    sched_cfg = DDPMScheduler.from_pretrained(cfg.dir / "model", subfolder="scheduler").config
    return unet, sched_cfg

In [ ]:
def train_run(cfg: RunCfg):
    run_dir = cfg.dir
    done_flag = run_dir / "train_info.json"
    if done_flag.exists():
        print(f"[skip] {cfg.name} already trained")
        return json.load(open(done_flag))
    (run_dir / "progress").mkdir(parents=True, exist_ok=True)
    json.dump(asdict(cfg) | {"name": cfg.name}, open(run_dir / "config.json", "w"), indent=2)

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)
    codes = train_codes(cfg.case, cfg.n_train)
    data = load_images(cfg.case, codes, cfg.image_size).to(DEVICE)

    model = build_model(cfg).to(DEVICE)
    ema = copy.deepcopy(model).eval().requires_grad_(False) if cfg.ema else None
    sched = build_train_scheduler(cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    lr_sched = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, cfg.steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    n_params = sum(p.numel() for p in model.parameters())

    start, losses, elapsed = 0, [], 0.0
    ckpt_path = run_dir / "ckpt.pt"
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        lr_sched.load_state_dict(ck["lr"]); scaler.load_state_dict(ck["scaler"])
        if ema is not None: ema.load_state_dict(ck["ema"])
        start, losses, elapsed = ck["step"], ck["losses"], ck["elapsed"]
        print(f"[resume] {cfg.name} from step {start}")

    print(f"[train] {cfg.name}: {len(codes)} signs, {cfg.steps} steps, {n_params/1e6:.1f}M params")
    gen = torch.Generator(device=DEVICE).manual_seed(cfg.seed * 100003 + start)
    T = cfg.num_train_timesteps
    model.train()
    t0 = time.time()
    for step in range(start, cfg.steps):
        idx = torch.randint(0, len(data), (cfg.batch_size,), device=DEVICE, generator=gen)
        clean = to_model(data[idx])
        noise = torch.randn(clean.shape, device=DEVICE, generator=gen)
        t = torch.randint(0, T, (clean.shape[0],), device=DEVICE, generator=gen)
        noisy = sched.add_noise(clean, noise, t)
        target = noise if cfg.prediction_type == "epsilon" else sched.get_velocity(clean, noise, t)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            pred = model(noisy, t, return_dict=False)[0]
        loss = F.mse_loss(pred.float(), target)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); lr_sched.step()
        if ema is not None:
            with torch.no_grad():
                for pe, p in zip(ema.parameters(), model.parameters()):
                    pe.lerp_(p, 1 - cfg.ema_decay)
                for be, b in zip(ema.buffers(), model.buffers()):
                    be.copy_(b)
        losses.append((step + 1, float(loss.item()), lr_sched.get_last_lr()[0]))

        if step + 1 - start == 50:
            rate = 50 / (time.time() - t0)
            print(f"   {rate:.2f} steps/s -> about {(cfg.steps - step) / rate / 60:.0f} min left for this run")
        if (step + 1) % 500 == 0:
            print(f"   step {step+1}/{cfg.steps}  loss(avg last 500)={np.mean([l for _, l, _ in losses[-500:]]):.4f}")
        if (step + 1) % SNAPSHOT_EVERY == 0 or step + 1 == cfg.steps:
            net = ema if ema is not None else model
            snap = sample(net, sched.config, 16, seed=EVAL_SEED, batch=16)
            grid_image(snap, 4, 4).save(run_dir / "progress" / f"step_{step+1:06d}.png")
            model.train()
        if (step + 1) % CKPT_EVERY == 0 and step + 1 < cfg.steps:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), lr=lr_sched.state_dict(),
                            scaler=scaler.state_dict(), ema=ema.state_dict() if ema is not None else None,
                            step=step + 1, losses=losses, elapsed=elapsed + time.time() - t0), ckpt_path)

    elapsed += time.time() - t0
    final = ema if ema is not None else model
    DDPMPipeline(unet=final, scheduler=sched).save_pretrained(run_dir / "model")
    pd.DataFrame(losses, columns=["step", "loss", "lr"]).to_csv(run_dir / "loss.csv", index=False)
    info = dict(name=cfg.name, n_train=len(codes), steps=cfg.steps, params_M=n_params / 1e6,
                train_minutes=elapsed / 60, epochs_equivalent=cfg.steps * cfg.batch_size / len(codes),
                final_loss=float(np.mean([l for _, l, _ in losses[-500:]])),
                gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
    json.dump(info, open(done_flag, "w"), indent=2)
    if ckpt_path.exists():
        ckpt_path.unlink()
    del model, ema, opt, data
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return info

## 3. Evaluation metrics

For each run, `N_EVAL_SAMPLES` images are generated from a fixed seed and compared with the **held-out test signs**, which the model never saw.

| Metric | Meaning | Better |
|---|---|---|
| **KID** (×10³) | Kernel Inception Distance between generated and real test images. More reliable than FID for small sets. | lower |
| **FID** | Fréchet Inception Distance (same comparison, biased at small n; report for convention) | lower |
| **Reference KID/FID** | Real training signs vs. real test signs: the best a perfect model could score | the floor |
| **Copy rate** | Fraction of generated images closer to a training sign than 95% of *unseen real* signs are. In other words, near-copies of the training data. | low (but not 0 if model is good at style) |
| **NN ratio** | Median nearest-training distance of generated images ÷ that of real test signs. Below 1 means memorizing; about 1 means new signs as different as real unseen signs. | ≈ 1 |
| **Ink / components distance** | Wasserstein distance between generated and real distributions of ink ratio and connected-component count (stroke weight and fragmentation) | lower |
| **Failure rate** | Blank or noise images (ink < 0.5% or > 60%) | lower |

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from scipy import ndimage, stats

def inception_metrics(real_u8, fake_u8):
    real_u8, fake_u8 = torch.as_tensor(real_u8), torch.as_tensor(fake_u8)
    sub = min(100, len(real_u8), len(fake_u8))
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    kid = KernelInceptionDistance(subset_size=sub, subsets=100, normalize=True).to(DEVICE)
    for arr, real in ((real_u8, True), (fake_u8, False)):
        for i in range(0, len(arr), 100):
            x = arr[i:i + 100].float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)
            fid.update(x, real=real); kid.update(x, real=real)
    km, ks = kid.compute()
    out = dict(fid=float(fid.compute()), kid_x1e3=float(km) * 1e3, kid_std_x1e3=float(ks) * 1e3)
    del fid, kid
    return out

def _nn_feats(u8, size=32):
    x = (255 - torch.as_tensor(u8).float()).div(255).unsqueeze(1).to(DEVICE)   # ink = 1
    x = F.interpolate(x, size=(size, size), mode="area")                         # light blur, shift tolerance
    return x.flatten(1)

def nn_analysis(gen_u8, train_u8, test_u8):
    G, Tr, Te = _nn_feats(gen_u8), _nn_feats(train_u8), _nn_feats(test_u8)
    D = G.shape[1] ** 0.5
    d_gen, idx = torch.cdist(G, Tr).min(1)
    d_test = torch.cdist(Te, Tr).min(1).values
    d_gen, d_test = (d_gen / D).cpu().numpy(), (d_test / D).cpu().numpy()
    thr = float(np.quantile(d_test, 0.05))
    return dict(copy_rate=float((d_gen < thr).mean()), copy_threshold=thr,
                nn_dist_gen_median=float(np.median(d_gen)), nn_dist_test_median=float(np.median(d_test)),
                nn_ratio=float(np.median(d_gen) / np.median(d_test))), idx.cpu().numpy(), d_gen, d_test

def shape_stats(u8):
    u8 = np.asarray(u8)
    ink = (u8 < 128)
    ratio = ink.reshape(len(u8), -1).mean(1)
    comps = np.array([ndimage.label(m)[1] for m in ink])
    return ratio, comps

def compare_shapes(gen_u8, real_u8):
    gr, gc = shape_stats(gen_u8); rr, rc = shape_stats(real_u8)
    return dict(ink_gen_mean=float(gr.mean()), ink_real_mean=float(rr.mean()),
                ink_wasserstein=float(stats.wasserstein_distance(gr, rr)),
                cc_gen_mean=float(gc.mean()), cc_real_mean=float(rc.mean()),
                cc_wasserstein=float(stats.wasserstein_distance(gc, rc)),
                failure_rate=float(((gr < 0.005) | (gr > 0.60)).mean()))

_REF_CACHE = {}
def reference_metrics(case, size):
    # Real train vs. real test: the best achievable score, used as the floor in plots.
    key = (case, size)
    path = REPORT_DIR / f"reference_{case}_{size}.json"
    if key in _REF_CACHE: return _REF_CACHE[key]
    if path.exists():
        _REF_CACHE[key] = json.load(open(path)); return _REF_CACHE[key]
    tr = train_codes(case)
    pick = list(np.random.default_rng(SEED).choice(tr, size=min(N_EVAL_SAMPLES, len(tr)), replace=False))
    ref = inception_metrics(load_images(case, test_codes(case), size), load_images(case, pick, size))
    json.dump(ref, open(path, "w"), indent=2)
    _REF_CACHE[key] = ref
    return ref

def nn_figure(gen, train, idx, d_gen, path, k=10):
    order = np.argsort(d_gen)
    rnd = np.random.default_rng(0).choice(len(gen), size=k, replace=False)
    fig, axes = plt.subplots(4, k, figsize=(1.2 * k, 5.2))
    for j in range(k):
        for r, (src, lab) in enumerate([(gen[order[j]], "closest\ngenerated"), (train[idx[order[j]]], "its nearest\ntraining sign"),
                                        (gen[rnd[j]], "random\ngenerated"), (train[idx[rnd[j]]], "its nearest\ntraining sign")]):
            axes[r, j].imshow(src, cmap="gray", vmin=0, vmax=255); axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
            if j == 0: axes[r, j].set_ylabel(lab, fontsize=8)
        axes[0, j].set_title(f"d={d_gen[order[j]]:.3f}", fontsize=7)
        axes[2, j].set_title(f"d={d_gen[rnd[j]]:.3f}", fontsize=7)
    fig.savefig(path); plt.close(fig)

def evaluate_run(cfg: RunCfg, force=False):
    mpath = cfg.dir / "metrics.json"
    if mpath.exists() and not force:
        return json.load(open(mpath))
    print(f"[eval] {cfg.name}")
    unet, sched_cfg = load_unet(cfg)
    t0 = time.time()
    gen = sample(unet, sched_cfg, N_EVAL_SAMPLES)
    sample_sec = (time.time() - t0) / N_EVAL_SAMPLES
    np.savez_compressed(cfg.dir / "samples.npz", gen=gen)
    grid_image(gen[:64], 8, 8).save(cfg.dir / "grid_final.png")

    test = load_images(cfg.case, test_codes(cfg.case), cfg.image_size).numpy()
    train = load_images(cfg.case, train_codes(cfg.case, cfg.n_train), cfg.image_size).numpy()
    m = inception_metrics(test, gen)
    nn, idx, d_gen, d_test = nn_analysis(gen, train, test)
    np.savez_compressed(cfg.dir / "nn.npz", idx=idx, d_gen=d_gen, d_test=d_test)
    nn_figure(gen, train, idx, d_gen, cfg.dir / "nn_pairs.png")
    ref = reference_metrics(cfg.case, cfg.image_size)
    info = json.load(open(cfg.dir / "train_info.json"))
    metrics = (asdict(cfg) | {"name": cfg.name} | info | m | nn | compare_shapes(gen, test)
               | {"ref_fid": ref["fid"], "ref_kid_x1e3": ref["kid_x1e3"], "sec_per_sample": sample_sec})
    json.dump(metrics, open(mpath, "w"), indent=2)
    if not SAVE_MODEL:
        shutil.rmtree(cfg.dir / "model", ignore_errors=True)
    del unet
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return metrics

## 4. Experiment plan

* **Q1:** each case × each size in `SUBSET_SIZES`, all with the baseline settings.
* **Q3:** changes on top of the full-data baseline. Change one thing at a time, plus one combination.

Runs that are already finished are skipped, so you can run this cell repeatedly across Colab sessions. Change `RUN_GROUPS` / `RUN_CASES` to run part of the plan. The first run prints its speed and estimated time.

**Rough budget:** the speed is printed after 50 steps. Multiply by the number of runs. The baseline UNet has 77M parameters, so on a T4 at 128 px expect **roughly an hour per 10k-step run**. To cut the cost: lower `TRAIN_STEPS` (keep it the same for every run), drop a subset size, run Q3 on one case only (`Q3_CASES`), or use an L4/A100 runtime. Run Q1 first: it is the core result.

In [ ]:
Q3_VARIANTS = {   # name -> settings changed from the baseline
    "cosine":     dict(beta_schedule="squaredcos_cap_v2"),
    "ema":        dict(ema=True),
    "cosine_ema": dict(beta_schedule="squaredcos_cap_v2", ema=True),
    "vpred_cos":  dict(beta_schedule="squaredcos_cap_v2", prediction_type="v_prediction"),
    "small_unet": dict(model="small"),
    "res64":      dict(image_size=64),
}
Q1_CASES = THIS_NOTEBOOK_CASES
Q3_CASES = THIS_NOTEBOOK_CASES     # e.g. ["hiero_normal"] to halve the Q3 cost

EXPERIMENTS = []
for case in Q1_CASES:
    for n in SUBSET_SIZES:
        EXPERIMENTS.append(RunCfg(case=case, n_train=n, group="Q1"))
for case in Q3_CASES:
    for v, kw in Q3_VARIANTS.items():
        EXPERIMENTS.append(RunCfg(case=case, n_train=None, group="Q3", variant=v, **kw))

print(f"{len(EXPERIMENTS)} runs planned")
for c in EXPERIMENTS:
    status = "done" if (c.dir / "metrics.json").exists() else ("trained" if (c.dir / "train_info.json").exists() else "todo")
    print(f"  [{status:7s}] {c.group}  {c.name}")

In [ ]:
RUN_GROUPS = ["Q1", "Q3"]          # run Q1 first: it gives you the core results
RUN_CASES  = THIS_NOTEBOOK_CASES

for cfg in EXPERIMENTS:
    if cfg.group not in RUN_GROUPS or cfg.case not in RUN_CASES:
        continue
    train_run(cfg)
    m = evaluate_run(cfg)
    print(f"   -> KID={m['kid_x1e3']:.2f}e-3  FID={m['fid']:.1f}  copy={m['copy_rate']:.1%}  NN-ratio={m['nn_ratio']:.2f}")

## 5. Results table

In [ ]:
def collect_results():
    rows = [json.load(open(p)) for p in RUNS_DIR.glob("*/metrics.json")]
    if not rows:
        raise RuntimeError("No evaluated runs yet: run Section 4 first.")
    df = pd.DataFrame(rows)
    df = df[df.case.isin(CASES)]   # every case run so far, from either notebook
    df["n_train"] = df["n_train"].astype(float)
    df["case_label"] = df.case.map(lambda c: CASES.get(c, {}).get("label", c))
    return df

results = collect_results()
results.to_csv(REPORT_DIR / "results.csv", index=False)

cols = ["case_label", "group", "variant", "n_train", "kid_x1e3", "kid_std_x1e3", "fid", "ref_kid_x1e3",
        "copy_rate", "nn_ratio", "ink_wasserstein", "cc_wasserstein", "failure_rate", "train_minutes", "epochs_equivalent"]
table = results.sort_values(["group", "case", "variant", "n_train"])[cols].round(3)
(REPORT_DIR / "results_table.md").write_text(table.to_markdown(index=False))
table

## 6. Q1: How much data is needed?

In [ ]:
q1 = results[results.group == "Q1"].sort_values("n_train")
colors = dict(zip(ALL_CASES, plt.cm.tab10.colors))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for case, d in q1.groupby("case"):
    c, lab = colors[case], CASES[case]["label"]
    axes[0].errorbar(d.n_train, d.kid_x1e3, yerr=d.kid_std_x1e3, marker="o", color=c, label=lab, capsize=3)
    axes[0].axhline(d.ref_kid_x1e3.iloc[0], color=c, ls="--", lw=1)
    axes[1].plot(d.n_train, d.copy_rate * 100, marker="o", color=c, label=lab)
    axes[2].plot(d.n_train, d.nn_ratio, marker="o", color=c, label=lab)
axes[0].set_ylabel("KID ×10³ (lower = better)\n dashed = real-vs-real floor")
axes[1].set_ylabel("copy rate (%)")
axes[2].set_ylabel("NN ratio (1 = as novel as real signs)"); axes[2].axhline(1, color="gray", ls=":")
for a in axes:
    a.set_xscale("log"); a.set_xlabel("training signs (log scale)"); a.grid(alpha=.3)
axes[0].legend()
fig.suptitle(f"Q1: effect of dataset size (fixed {TRAIN_STEPS} training steps)")
savefig(fig, "fig_q1_dataset_size.png")

In [ ]:
# Samples from the SAME starting noise for each dataset size: rows = size, columns = case
sizes = sorted(q1.n_train.unique())
cases_q1 = [c for c in CASES if c in set(q1.case)]
fig, axes = plt.subplots(len(sizes), len(cases_q1), figsize=(3.2 * len(cases_q1), 3.2 * len(sizes)), squeeze=False)
for i, n in enumerate(sizes):
    for j, case in enumerate(cases_q1):
        ax = axes[i, j]; ax.axis("off")
        row = q1[(q1.case == case) & (q1.n_train == n)]
        if len(row):
            gen = np.load(RUNS_DIR / row.name.iloc[0] / "samples.npz")["gen"]
            ax.imshow(grid_image(gen[:16], 4, 4), cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{CASES[case]['label']}\nN={int(n)}  KID={row.kid_x1e3.iloc[0]:.1f}e-3", fontsize=9)
fig.suptitle("Q1: generated samples vs. dataset size (same noise seeds)")
savefig(fig, "fig_q1_samples_by_size.png")

In [ ]:
# Training loss curves (smoothed)
fig, axes = plt.subplots(1, len(cases_q1), figsize=(6 * len(cases_q1), 3.5), squeeze=False)
for j, case in enumerate(cases_q1):
    ax = axes[0, j]
    for _, r in q1[q1.case == case].iterrows():
        L = pd.read_csv(RUNS_DIR / r["name"] / "loss.csv")
        ax.plot(L.step, L.loss.rolling(200, min_periods=1).mean(), label=f"N={int(r.n_train)}")
    ax.set_title(CASES[case]["label"]); ax.set_xlabel("step"); ax.set_ylabel("MSE loss (smoothed)")
    ax.set_yscale("log"); ax.grid(alpha=.3); ax.legend(fontsize=8)
savefig(fig, "fig_q1_loss_curves.png")

## 7. Q2: Quality and meaningfulness
The "best run" for each case is the one with the lowest KID among the **Q1** runs.

In [ ]:
best = q1.loc[q1.groupby("case").kid_x1e3.idxmin()].set_index("case")
print(best[["name", "n_train", "kid_x1e3", "fid", "copy_rate", "nn_ratio"]])
best_cfg = {case: RunCfg(case=case, **{k: best.loc[case][k] for k in RunCfg.__dataclass_fields__ if k in best.columns and k not in ("n_train", "case")},
                         n_train=None if pd.isna(best.loc[case].n_train) or int(best.loc[case].n_train) == len(train_codes(case)) else int(best.loc[case].n_train))
            for case in best.index}
for case, c in best_cfg.items():
    assert c.dir.name == best.loc[case]["name"], (c.dir.name, best.loc[case]["name"])

# Real (held-out) vs generated
fig, axes = plt.subplots(len(best), 2, figsize=(9, 4.6 * len(best)), squeeze=False)
for i, case in enumerate(best.index):
    test = load_images(case, test_codes(case), int(best.loc[case].image_size)).numpy()
    gen = np.load(RUNS_DIR / best.loc[case]["name"] / "samples.npz")["gen"]
    pick = np.random.default_rng(1).choice(len(test), size=min(36, len(test)), replace=False)
    axes[i, 0].imshow(grid_image(test[pick], 6, 6), cmap="gray", vmin=0, vmax=255)
    axes[i, 1].imshow(grid_image(gen[:36], 6, 6), cmap="gray", vmin=0, vmax=255)
    axes[i, 0].set_title(f"{CASES[case]['label']}: real (held-out)"); axes[i, 1].set_title("generated")
    for a in axes[i]: a.axis("off")
savefig(fig, "fig_q2_real_vs_generated.png")

In [ ]:
# Novelty: nearest training sign for generated images (one figure per case)
for case in best.index:
    p = RUNS_DIR / best.loc[case]["name"] / "nn_pairs.png"
    shutil.copy(p, FIG_DIR / f"fig_q2_nearest_neighbours_{case}.png")
    display(Image.open(p))

# Distribution of nearest-training distances: generated vs. real unseen signs
fig, axes = plt.subplots(1, len(best), figsize=(5.5 * len(best), 3.4), squeeze=False)
for j, case in enumerate(best.index):
    z = np.load(RUNS_DIR / best.loc[case]["name"] / "nn.npz")
    ax = axes[0, j]
    bins = np.linspace(0, max(z["d_gen"].max(), z["d_test"].max()), 40)
    ax.hist(z["d_test"], bins=bins, alpha=.6, density=True, label="real unseen signs")
    ax.hist(z["d_gen"], bins=bins, alpha=.6, density=True, label="generated")
    ax.axvline(best.loc[case].copy_threshold, color="k", ls="--", lw=1, label="copy threshold")
    ax.set_title(f"{CASES[case]['label']}  (copy rate {best.loc[case].copy_rate:.1%})")
    ax.set_xlabel("distance to nearest training sign"); ax.legend(fontsize=8)
savefig(fig, "fig_q2_novelty_hist.png")

# Shape statistics: ink ratio and fragmentation, generated vs real
fig, axes = plt.subplots(2, len(best), figsize=(5.5 * len(best), 6), squeeze=False)
for j, case in enumerate(best.index):
    test = load_images(case, test_codes(case), int(best.loc[case].image_size)).numpy()
    gen = np.load(RUNS_DIR / best.loc[case]["name"] / "samples.npz")["gen"]
    (gr, gc), (rr, rc) = shape_stats(gen), shape_stats(test)
    axes[0, j].hist(rr, bins=40, alpha=.6, density=True, label="real"); axes[0, j].hist(gr, bins=40, alpha=.6, density=True, label="generated")
    axes[0, j].set_xlabel("ink ratio"); axes[0, j].set_title(CASES[case]["label"]); axes[0, j].legend()
    hi = int(np.percentile(np.concatenate([rc, gc]), 99)) + 2
    axes[1, j].hist(rc, bins=range(0, hi), alpha=.6, density=True, label="real"); axes[1, j].hist(gc, bins=range(0, hi), alpha=.6, density=True, label="generated")
    axes[1, j].set_xlabel("connected components per image (fragmentation)"); axes[1, j].legend()
savefig(fig, "fig_q2_shape_stats.png")

In [ ]:
# Training progress: the same noise seeds sampled at different training steps
for case in best.index:
    snaps = sorted((RUNS_DIR / best.loc[case]["name"] / "progress").glob("step_*.png"))
    if not snaps: continue
    pick = [snaps[i] for i in np.unique(np.linspace(0, len(snaps) - 1, min(6, len(snaps))).astype(int))]
    fig, axes = plt.subplots(1, len(pick), figsize=(2.6 * len(pick), 3), squeeze=False)
    for a, p in zip(axes[0], pick):
        a.imshow(Image.open(p), cmap="gray", vmin=0, vmax=255); a.axis("off")
        a.set_title(f"step {int(p.stem.split('_')[1])}")
    fig.suptitle(f"{CASES[case]['label']}: training progress (fixed noise)")
    savefig(fig, f"fig_q2_training_progress_{case}.png")

In [ ]:
# True denoising trajectory (x_t at selected timesteps) + GIF
import imageio.v2 as imageio
for case, cfg in best_cfg.items():
    if not (cfg.dir / "model").exists():
        print("model not saved for", cfg.name, "- skipping trajectory"); continue
    unet, sched_cfg = load_unet(cfg)
    _, traj = sample(unet, sched_cfg, 1, seed=7, batch=1, keep_traj=True)
    _, traj2 = sample(unet, sched_cfg, 1, seed=11, batch=1, keep_traj=True)
    sel = np.unique(np.linspace(0, len(traj) - 1, 8).astype(int))
    fig, axes = plt.subplots(2, len(sel), figsize=(1.8 * len(sel), 4))
    for r, tr in enumerate([traj, traj2]):
        for a, k in zip(axes[r], sel):
            a.imshow(tr[k][1], cmap="gray", vmin=0, vmax=255); a.axis("off")
            if r == 0: a.set_title(f"t={tr[k][0]}", fontsize=9)
    fig.suptitle(f"{CASES[case]['label']}: denoising trajectory ({INFERENCE_STEPS} DPM-Solver++ steps)")
    savefig(fig, f"fig_q2_denoising_{case}.png")
    frames = [np.array(Image.fromarray(f).resize((256, 256), Image.NEAREST)) for _, f in traj]
    imageio.mimsave(FIG_DIR / f"denoising_{case}.gif", frames + [frames[-1]] * 10, duration=0.12, loop=0)
    del unet

### Human evaluation pack
For each case, 30 real test signs and 30 generated signs are shuffled and numbered. Share `contact_sheet_<case>.png` and `rating_sheet_<case>.csv` with 3–5 people. For each image they record **plausibility (1–5)** and **real or generated?**. Keep `answer_key_<case>.csv` hidden. The next cell scores the returned sheets.

In [ ]:
HE_DIR = REPORT_DIR / "human_eval"; HE_DIR.mkdir(exist_ok=True)
N_HE = 30
for case in best.index:
    rng = np.random.default_rng(99)
    size = int(best.loc[case].image_size)
    test = load_images(case, test_codes(case), size).numpy()
    gen = np.load(RUNS_DIR / best.loc[case]["name"] / "samples.npz")["gen"]
    real = test[rng.choice(len(test), N_HE, replace=False)]
    fake = gen[rng.choice(len(gen), N_HE, replace=False)]
    if real.shape[1] != fake.shape[1]:
        fake = np.stack([np.asarray(Image.fromarray(f).resize(real.shape[1:][::-1], Image.LANCZOS)) for f in fake])
    imgs = np.concatenate([real, fake]); src = ["real"] * N_HE + ["generated"] * N_HE
    order = rng.permutation(len(imgs))
    cols = 10; rows_ = math.ceil(len(imgs) / cols); h = imgs.shape[1]
    sheet = Image.new("L", (cols * (h + 8), rows_ * (h + 24)), 255); dr = ImageDraw.Draw(sheet)
    for k, o in enumerate(order):
        r, c = divmod(k, cols)
        sheet.paste(Image.fromarray(imgs[o]), (c * (h + 8), r * (h + 24) + 18))
        dr.text((c * (h + 8) + 2, r * (h + 24) + 2), f"#{k + 1}", fill=0)
    sheet.save(HE_DIR / f"contact_sheet_{case}.png")
    pd.DataFrame({"id": range(1, len(order) + 1), "plausibility_1to5": "", "real_or_generated": ""}).to_csv(HE_DIR / f"rating_sheet_{case}.csv", index=False)
    pd.DataFrame({"id": range(1, len(order) + 1), "source": [src[o] for o in order]}).to_csv(HE_DIR / f"answer_key_{case}.csv", index=False)
    print("saved human-eval pack for", case)

In [ ]:
# After collecting ratings: save each rater's sheet as rating_sheet_<case>_<rater>.csv in HE_DIR, then run this cell.
summ = []
for case in CASES:
    key_p = HE_DIR / f"answer_key_{case}.csv"
    if not key_p.exists(): continue
    key = pd.read_csv(key_p)
    for p in HE_DIR.glob(f"rating_sheet_{case}_*.csv"):
        r = pd.read_csv(p).merge(key, on="id")
        r["guess_correct"] = r.real_or_generated.str.strip().str.lower().str.startswith(r.source.str[0])
        g = r.groupby("source").plausibility_1to5.mean()
        summ.append(dict(case=case, rater=p.stem.split("_")[-1], plaus_real=g.get("real"), plaus_generated=g.get("generated"),
                         detection_accuracy=r.guess_correct.mean()))
if summ:
    he = pd.DataFrame(summ); he.to_csv(REPORT_DIR / "human_eval_results.csv", index=False)
    display(he.groupby("case").mean(numeric_only=True))   # detection accuracy near 50% = indistinguishable from real
else:
    print("No returned rating sheets yet.")

## 8. Q3: Modifications to the architecture and noise schedule

In [ ]:
q3_base = q1[q1.n_train == q1.groupby("case").n_train.transform("max")].assign(variant="baseline")
q3 = pd.concat([q3_base, results[results.group == "Q3"]])
variants = ["baseline"] + [v for v in Q3_VARIANTS if v in set(q3.variant)]
cases_q3 = [c for c in CASES if c in set(q3.case)]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
w = 0.8 / max(1, len(cases_q3)); x = np.arange(len(variants))
for j, case in enumerate(cases_q3):
    d = q3[q3.case == case].set_index("variant").reindex(variants)
    axes[0].bar(x + j * w, d.kid_x1e3, w, yerr=d.kid_std_x1e3, label=CASES[case]["label"], color=colors[case], capsize=2)
    axes[1].bar(x + j * w, d.copy_rate * 100, w, label=CASES[case]["label"], color=colors[case])
for a, yl in zip(axes, ["KID ×10³ (lower = better)", "copy rate (%)"]):
    a.set_xticks(x + w * (len(cases_q3) - 1) / 2); a.set_xticklabels(variants, rotation=20); a.set_ylabel(yl); a.grid(axis="y", alpha=.3)
axes[0].legend()
fig.suptitle("Q3: modifications vs. baseline (all training signs)")
savefig(fig, "fig_q3_ablations.png")

q3_table = q3[["case_label", "variant", "beta_schedule", "prediction_type", "ema", "model", "image_size",
               "kid_x1e3", "fid", "copy_rate", "nn_ratio", "failure_rate", "train_minutes"]].round(3)
(REPORT_DIR / "q3_table.md").write_text(q3_table.to_markdown(index=False))
q3_table

In [ ]:
# Sample grids: baseline vs each variant, same noise seeds
fig, axes = plt.subplots(len(cases_q3), len(variants), figsize=(2.7 * len(variants), 3 * len(cases_q3)), squeeze=False)
for i, case in enumerate(cases_q3):
    for j, v in enumerate(variants):
        ax = axes[i, j]; ax.axis("off")
        row = q3[(q3.case == case) & (q3.variant == v)]
        if len(row):
            gen = np.load(RUNS_DIR / row["name"].iloc[0] / "samples.npz")["gen"]
            ax.imshow(grid_image(gen[:9], 3, 3), cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{v}\nKID={row.kid_x1e3.iloc[0]:.1f}e-3", fontsize=8)
        if j == 0: ax.text(-0.08, 0.5, CASES[case]["label"], transform=ax.transAxes, rotation=90, va="center", ha="right")
savefig(fig, "fig_q3_samples_by_variant.png")

In [ ]:
# Sampling-steps ablation (no retraining): quality vs. number of DPM-Solver++ steps for the best run of each case
STEP_GRID = [5, 10, 25, 50, 100]
N_STEP_EVAL = min(300, N_EVAL_SAMPLES)
path = REPORT_DIR / "inference_steps.csv"
done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["case", "steps"])
rows = done.to_dict("records")
for case, cfg in best_cfg.items():
    if not (cfg.dir / "model").exists(): continue
    unet, sched_cfg = load_unet(cfg)
    test = load_images(case, test_codes(case), cfg.image_size).numpy()
    for s in STEP_GRID:
        if ((done.case == case) & (done.steps == s)).any(): continue
        t0 = time.time()
        gen = sample(unet, sched_cfg, N_STEP_EVAL, steps=s)
        rows.append(dict(case=case, steps=s, sec_per_image=(time.time() - t0) / N_STEP_EVAL,
                         **inception_metrics(test, gen), failure_rate=compare_shapes(gen, test)["failure_rate"]))
        pd.DataFrame(rows).to_csv(path, index=False)
    del unet
steps_df = pd.DataFrame(rows)
if len(steps_df):
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    for case, d in steps_df.groupby("case"):
        d = d.sort_values("steps")
        ax.errorbar(d.steps, d.kid_x1e3, yerr=d.kid_std_x1e3, marker="o", capsize=3, label=CASES[case]["label"], color=colors[case])
    ax.set_xscale("log"); ax.set_xlabel("sampling steps"); ax.set_ylabel("KID ×10³"); ax.grid(alpha=.3); ax.legend()
    ax.set_title("Q3: quality vs. number of sampling steps")
    savefig(fig, "fig_q3_inference_steps.png")
steps_df

## 9. Key numbers for the write-up

In [ ]:
lines = []
for case in best.index:
    b = best.loc[case]
    q1c = q1[q1.case == case].sort_values("n_train")
    lines.append(f"### {CASES[case]['label']}")
    lines.append(f"- Best Q1 run: N={int(b.n_train)} signs, KID={b.kid_x1e3:.2f}e-3 (real-vs-real floor {b.ref_kid_x1e3:.2f}e-3), FID={b.fid:.1f}")
    lines.append(f"- Copy rate {b.copy_rate:.1%}, NN ratio {b.nn_ratio:.2f}, failure rate {b.failure_rate:.1%}")
    lines.append("- KID by dataset size: " + ", ".join(f"N={int(n)}: {k:.2f}" for n, k in zip(q1c.n_train, q1c.kid_x1e3)))
    if len(q3) and case in set(q3.case):
        q3c = q3[q3.case == case].set_index("variant")
        top = q3c.kid_x1e3.idxmin()
        lines.append(f"- Best Q3 setting: {top} (KID {q3c.kid_x1e3[top]:.2f}e-3 vs baseline {q3c.kid_x1e3.get('baseline', float('nan')):.2f}e-3)")
    lines.append("")
if {"hiero_normal", "hiero_bold"} <= set(best.index):
    n, b = best.loc["hiero_normal"], best.loc["hiero_bold"]
    lines.append(f"### Normal vs Bold\n- KID normal {n.kid_x1e3:.2f}e-3 vs bold {b.kid_x1e3:.2f}e-3; "
                 f"components gap (Wasserstein) normal {n.cc_wasserstein:.2f} vs bold {b.cc_wasserstein:.2f}")
# Cross-script comparison at the dataset sizes both scripts were trained on
cases_now = list(q1.case.unique())
if len(cases_now) > 1:
    shared_n = sorted(set.intersection(*(set(q1[q1.case == c].n_train) for c in cases_now)))
    if shared_n:
        lines.append("")
        lines.append("### Same dataset size, different scripts (KID ×10³, lower is better)")
        for n_ in shared_n:
            lines.append(f"- N={int(n_)}: " + ", ".join(
                f"{CASES[c]['label']} {q1[(q1.case == c) & (q1.n_train == n_)].kid_x1e3.iloc[0]:.2f}" for c in cases_now))
summary_md = "\n".join(lines)
(REPORT_DIR / "key_numbers.md").write_text(summary_md)
print(summary_md)
print("\nAll figures:", *sorted(p.name for p in FIG_DIR.glob("*")), sep="\n  ")